# max-back-tied-half — faded example 2: Implement relu_back as a Specialization of maximum_back0

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `max-back-tied-half`. The last cell reports your progress on the `Backprop: max_back with tied half-mass` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: max_back with tied half-mass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`max-back-tied-half`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "max-back-tied-half"
DD_SUBTOPIC = "Backprop: max_back with tied half-mass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Since `relu(x) = maximum(x, 0)`, the relu backward is exactly `maximum_back0` with `y = zeros_like(x)`. At `x == 0` (the kink), the half-mass convention assigns `0.5 * grad_out` rather than the strict-inequality value of 0. This specialization shows that one general rule covers all elementwise-max operations.

## Faded exercise 2

Complete `relu_back` below using the provided `maximum_back0`. Fill in the single blanked line that calls `maximum_back0` with the correct second argument.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def relu_back(grad_out, x):
    return maximum_back0(grad_out, x, t.zeros_like(x))

# Exercise it
x = t.tensor([-2.0, 0.0, 1.5, -0.5, 3.0])
grad_out = t.ones(5)
g = relu_back(grad_out, x)
print(f"relu_back: {g.tolist()}")
print(f"Expected:  [0.0, 0.5, 1.0, 0.0, 1.0]")


import torch as t

def _test():
    x = t.tensor([-2.0, 0.0, 1.5, -0.5, 3.0])
    grad_out = t.ones(5)

    g = relu_back(grad_out, x)

    expected = t.tensor([0.0, 0.5, 1.0, 0.0, 1.0])
    assert t.allclose(g, expected, atol=1e-6), f"Got {g.tolist()}"

    # At positive positions, agree with PyTorch's strict convention
    pos = x > 0
    x_ag = x.clone().requires_grad_(True)
    t.relu(x_ag).sum().backward()
    assert t.allclose(g[pos], x_ag.grad[pos], atol=1e-6)

    # At kink, half-mass differs from strict
    assert abs(g[1].item() - 0.5) < 1e-6, "Half-mass must give 0.5 at kink"
    assert abs(x_ag.grad[1].item() - 0.0) < 1e-6, "PyTorch strict gives 0 at kink"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def relu_back(grad_out, x):
    return maximum_back0(grad_out, x, t.zeros_like(x))

# Exercise it
x = t.tensor([-2.0, 0.0, 1.5, -0.5, 3.0])
grad_out = t.ones(5)
g = relu_back(grad_out, x)
print(f"relu_back: {g.tolist()}")
print(f"Expected:  [0.0, 0.5, 1.0, 0.0, 1.0]")
```
</details>